In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

In [9]:
import os

print(os.path.exists(r"..\model\artifacts\finalfeature_cols.pkl"))

True


In [10]:
df = pd.read_csv(r'..\Dataset\water_quality_cleaneddata.csv')
model = joblib.load(r"..\model\artifacts\finalbest_model_tuned.pkl")
qt = joblib.load(r"..\model\artifacts\qt.pkl")
FEATURE_COLS = joblib.load(r"..\model\artifacts\finalfeature_cols.pkl")

print(f"Loaded cleaned data: {df.shape}")
print(f"Model expects {len(FEATURE_COLS)} features:", FEATURE_COLS)

d:\mini_water_predict\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Loaded cleaned data: (2617, 17)
Model expects 14 features: ['Temperature', 'DO_sqrt', 'pH', 'Conductivity_log', 'BOD_log', 'Nitrate_Nitrite_log', 'State_freq', 'Water_Body_Type_POND', 'Water_Body_Type_TANK', 'Water_Body_Type_WETLAND', 'BOD_Temp_log', 'BOD_Conductivity_log', 'Nitrate_Temp', 'Is_Monsoon']


d:\mini_water_predict\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreesRegressor from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\mini_water_predict\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator QuantileTransformer from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [11]:
# compare state-year averages for all features + target
BASE_COLS = ['Temperature', 'DO', 'pH', 'Conductivity', 'BOD', 'Nitrate_Nitrite']
TARGET_COL = 'Fecal_Coliform'

state_year = (
    df.groupby(['State_Name', 'Year'])[BASE_COLS + [TARGET_COL]]
    .mean()
    .reset_index()
)

print(f"State-year rows: {state_year.shape}")
state_year.head()

State-year rows: (138, 9)


,State_Name,Year,Temperature,DO,pH,Conductivity,BOD,Nitrate_Nitrite,Fecal_Coliform
0,ANDHRA \nPRADESH,2017,26.000000,6.350000,7.650000,815.500000,1.800000,2.260000,20.000000
1,ANDHRA \nPRADESH,2018,28.000000,6.250000,7.700000,962.500000,1.350000,2.000000,27.000000
2,ANDHRA \nPRADESH,2019,28.750000,5.275000,7.550000,997.500000,7.450000,2.555000,83.500000
3,ANDHRA \nPRADESH,2020,27.250000,5.600000,7.350000,586.500000,2.625000,2.240000,71.250000
4,ANDHRA \nPRADESH,2022,24.608696,5.854348,7.647826,912.086957,3.867391,1.661087,171.565217


In [ ]:
# === Cell 4: Helper — extrapolate one feature for one state to a target year ===
def extrapolate_feature(state_df, feature, target_year):
    """Fit Year -> feature on available years, predict at target_year."""
    sub = state_df.dropna(subset=[feature])
    if len(sub) < 2:
        # Not enough history to fit a trend — fall back to last known value
        return sub[feature].iloc[-1] if len(sub) == 1 else np.nan
    X = sub[['Year']].values
    y = sub[feature].values
    reg = LinearRegression().fit(X, y)
    pred = reg.predict([[target_year]])[0]
    return max(pred, 0)  # water quality features can't be negative

In [ ]:
# === Cell 5: Build synthetic state-year row for a target year ===
def build_synthetic_year(state_year_df, target_year, state_freq_mapping, water_type_mode):
    """
    state_year_df : state_year table filtered to years < target_year
    Returns a dataframe with one row per state, containing extrapolated
    BASE_COLS, ready for feature engineering.
    """
    rows = []
    for state in state_year_df['State_Name'].unique():
        state_hist = state_year_df[state_year_df['State_Name'] == state].sort_values('Year')
        row = {'State_Name': state, 'Year': target_year}
        for feat in BASE_COLS:
            row[feat] = extrapolate_feature(state_hist, feat, target_year)
        rows.append(row)

    synth_df = pd.DataFrame(rows)
    # Water_Body_Type: use the most common type historically per state (mode)
    synth_df['Water_Body_Type'] = synth_df['State_Name'].map(water_type_mode)
    return synth_df

In [ ]:
# === Cell 6: Feature engineering — MUST mirror the ML notebook exactly ===
def apply_feature_engineering(synth_df, state_freq_mapping):
    d = synth_df.copy()

    d['DO_sqrt'] = np.sqrt(d['DO'].clip(lower=0))
    d['Conductivity_log'] = np.log1p(d['Conductivity'])
    d['BOD_log'] = np.log1p(d['BOD'])
    d['Nitrate_Nitrite_log'] = np.log1p(d['Nitrate_Nitrite'])

    d['BOD_Temp'] = d['BOD'] * d['Temperature']
    d['BOD_Conductivity'] = d['BOD'] * d['Conductivity']
    d['Nitrate_Temp'] = d['Nitrate_Nitrite'] * d['Temperature']

    d['BOD_Temp_log'] = np.log1p(d['BOD_Temp'].clip(lower=0))
    d['BOD_Conductivity_log'] = np.log1p(d['BOD_Conductivity'].clip(lower=0))

    d['Is_Monsoon'] = d['Year'].astype(int).apply(
        lambda x: 1 if x in [2017, 2018, 2019, 2020, 2021, 2022, 2023] else 0
    )

    d['State_freq'] = d['State_Name'].map(state_freq_mapping).fillna(0)

    d = pd.get_dummies(d, columns=['Water_Body_Type'])
    # Ensure every dummy column the model expects exists, even if absent this round
    for col in FEATURE_COLS:
        if col.startswith('Water_Body_Type_') and col not in d.columns:
            d[col] = 0

    return d

In [ ]:
# === Cell 7: Precompute static mappings (state frequency + dominant water type) ===
state_freq_mapping = df['State_Name'].value_counts(normalize=True).to_dict()
water_type_mode = df.groupby('State_Name')['Water_Body_Type'].mode().apply(lambda x: x[0]).to_dict()

print("State frequency mapping ready:", len(state_freq_mapping), "states")
print("Dominant water type per state ready.")

In [ ]:
# === Cell 8: Walk-forward loop ===
results = []

all_years = sorted(state_year['Year'].unique())
forecast_years = list(range(min(all_years) + 1, 2024))  # e.g. 2018..2023

for target_year in forecast_years:
    train_hist = state_year[state_year['Year'] < target_year]

    if train_hist.empty:
        continue

    synth = build_synthetic_year(train_hist, target_year, state_freq_mapping, water_type_mode)
    synth_fe = apply_feature_engineering(synth, state_freq_mapping)

    X_pred = synth_fe.reindex(columns=FEATURE_COLS, fill_value=0)

    pred_qt = model.predict(X_pred)
    pred_raw = qt.inverse_transform(pred_qt.reshape(-1, 1)).flatten().clip(0)

    synth['Predicted_FC'] = pred_raw
    synth['Year'] = target_year

    # Attach real value if it exists (years <= 2022)
    real_lookup = (
        state_year[state_year['Year'] == target_year]
        .set_index('State_Name')
        .get('Fecal_Coliform') if 'Fecal_Coliform' in state_year.columns else None
    )
    # NOTE: Fecal_Coliform wasn't in BASE_COLS above — see Cell 3 fix below

    synth['Safety'] = synth['Predicted_FC'].apply(lambda x: 'Safe' if x <= 50 else 'Not Safe')
    results.append(synth[['State_Name', 'Year', 'Predicted_FC', 'Safety']])

forecast_all = pd.concat(results, ignore_index=True)
print(f"Forecast rows: {forecast_all.shape}")
forecast_all.head(10)

In [ ]:
# === Cell 9: Accuracy summary per year (only where real data exists) ===
eval_df = forecast_all.dropna(subset=['Real_FC'])

accuracy_summary = (
    eval_df.groupby('Year')
    .apply(lambda g: pd.Series({
        'MAE': (g['Predicted_FC'] - g['Real_FC']).abs().mean(),
        'States_Evaluated': len(g)
    }))
    .reset_index()
)

print(accuracy_summary)

In [ ]:
# === Cell 10: Save final output ===
forecast_all.to_csv("data/forecast_all_years.csv", index=False)
print("Saved: data/forecast_all_years.csv")
print(f"Years covered: {sorted(forecast_all['Year'].unique())}")